In [1]:
import requests
import pandas as pd

# Coordinates for NMDC Bailadila Iron Ore Mines (Bacheli/Kirandul)
lat = 18.7311
lon = 81.2855

# Open-Meteo Historical API URL (Pulling data for the 2023 Monsoon season: July to Oct)
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2023-07-01&end_date=2023-10-31&hourly=temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m,visibility&timezone=Asia%2FKolkata"

response = requests.get(url)
data = response.json()

# Convert to a clean Pandas DataFrame
df = pd.DataFrame(data['hourly'])
df['time'] = pd.to_datetime(df['time'])
df.rename(columns={
    'temperature_2m': 'Temp_C',
    'relative_humidity_2m': 'Humidity_%',
    'dew_point_2m': 'Dew_Point_C',
    'wind_speed_10m': 'Wind_Speed_kmh',
    'visibility': 'Visibility_m'
}, inplace=True)

# Save to CSV to use in your Streamlit dashboard
df.to_csv("Bailadila_Monsoon_Fog_Data.csv", index=False)
print("Dataset successfully generated for Bailadila!")

Dataset successfully generated for Bailadila!


In [2]:
import pandas as pd
import numpy as np

# Load the dataset from your VS Code environment
df = pd.read_csv("Bailadila_Monsoon_Fog_Data.csv")

# 1. Handle Missing Visibility (Synthetic Proxy for Prototype)
# If Open-Meteo returned nulls for visibility, generate realistic target values 
# based on the monsoon humidity conditions to train the model.
if df['Visibility_m'].isnull().all():
    df['Visibility_m'] = np.where(
        df['Humidity_%'] >= 90, 
        np.random.uniform(3, 50, len(df)),     # Dense fog (Bailadila conditions)
        np.where(
            df['Humidity_%'] >= 80,
            np.random.uniform(50, 200, len(df)), # Moderate haze
            np.random.uniform(1000, 5000, len(df)) # Clear visibility
        )
    )

# 2. Engineer Top Predictors
# Dew-point deficit is a highly critical feature for RF/XGBoost models
df['Dew_Point_Deficit'] = df['Temp_C'] - df['Dew_Point_C']

# 3. Engineer Time-Lagged Features
# Creating a 1-hour and 2-hour lookback window for the sequential dynamics
df['Visibility_Lag_1h'] = df['Visibility_m'].shift(1)
df['Humidity_Lag_1h'] = df['Humidity_%'].shift(1)
df['Dew_Point_Deficit_Lag_1h'] = df['Dew_Point_Deficit'].shift(1)

# Drop the initial rows with NaN values caused by the shift() operation
df = df.dropna()

# 4. Define Training Matrices
features = [
    'Temp_C', 'Humidity_%', 'Wind_Speed_kmh', 
    'Dew_Point_Deficit', 'Visibility_Lag_1h', 
    'Humidity_Lag_1h', 'Dew_Point_Deficit_Lag_1h'
]

X = df[features]
y = df['Visibility_m']

print(f"Feature matrix shape ready for XGBoost: {X.shape}")

Feature matrix shape ready for XGBoost: (2951, 7)


In [5]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBRegressor, XGBClassifier
import joblib

# ---------------------------------------------------------------------------
# 0. Config
# ---------------------------------------------------------------------------
FOG_THRESHOLD_CRITICAL = 3.0
FOG_THRESHOLD_ADVISORY = 10.0
QUANTILES = [0.1, 0.5, 0.9]
HORIZONS = [1, 2, 3]
CLASS_LABELS = {0: "CLEAR", 1: "ADVISORY", 2: "CRITICAL"}

# ---------------------------------------------------------------------------
# 1. Load & Clean
# ---------------------------------------------------------------------------
df = pd.read_csv("Bailadila_Monsoon_Fog_Data.csv")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time").sort_index()

rename_map = {
    "Temp_C": "Temp_C",
    "Humidity_%": "Rel Hum_%",
    "Dew_Point_C": "Dew Point Temp_C",
    "Wind_Speed_kmh": "Wind Speed_km/h"
}
df.rename(columns=rename_map, inplace=True)

# Drop the raw empty Visibility_m column so it doesn't cause df.dropna() to delete all rows
df = df.drop(columns=["Visibility_m", "Weather"], errors="ignore")

# Generate synthetic Visibility_km target based on monsoon meteorological rules
dew_deficit = np.maximum(df["Temp_C"] - df["Dew Point Temp_C"], 0.0)
conditions = [
    (df["Rel Hum_%"] >= 92) & (dew_deficit <= 1.5) & (df["Wind Speed_km/h"] <= 8.0),
    (df["Rel Hum_%"] >= 80) & (dew_deficit <= 3.0),
    (df["Rel Hum_%"] >= 70)
]
choices = [
    np.random.uniform(0.1, 2.8, len(df)),
    np.random.uniform(3.0, 9.5, len(df)),
    np.random.uniform(10.0, 20.0, len(df))
]
df["Visibility_km"] = np.select(conditions, choices, default=np.random.uniform(22.0, 25.0, len(df)))

if "Press_kPa" not in df.columns:
    df["Press_kPa"] = 101.3 - (df["Temp_C"] * 0.05) + np.random.normal(0, 0.1, len(df))

# ---------------------------------------------------------------------------
# 2. Feature Engineering
# ---------------------------------------------------------------------------
df["Dew_Point_Deficit"] = np.maximum(df["Temp_C"] - df["Dew Point Temp_C"], 0.0)
df["Fog_Stability_Index"] = df["Rel Hum_%"] / (1.0 + df["Wind Speed_km/h"] + df["Dew_Point_Deficit"])
df["Temp_Drop_1h"] = df["Temp_C"].diff(1)
df["Press_Drop_3h"] = df["Press_kPa"].diff(3)

df["Vis_RoC_1h"] = df["Visibility_km"].diff(1)
df["Vis_RoC_3h"] = df["Visibility_km"].diff(3)

_fog_now = (df["Visibility_km"] <= FOG_THRESHOLD_ADVISORY).astype(int)
_run_id = (_fog_now != _fog_now.shift()).cumsum()
df["Consec_Fog_Hours"] = (_fog_now.groupby(_run_id).cumcount() + 1) * _fog_now

df["Hour_Sin"] = np.sin(2 * np.pi * df.index.hour / 24)
df["Hour_Cos"] = np.cos(2 * np.pi * df.index.hour / 24)
df["Month_Sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["Month_Cos"] = np.cos(2 * np.pi * df.index.month / 12)

target_cols = ["Temp_C", "Rel Hum_%", "Visibility_km", "Press_kPa"]
for col in target_cols:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag2"] = df[col].shift(2)
    df[f"{col}_lag3"] = df[col].shift(3)
    df[f"{col}_roll_mean_3h"] = df[col].rolling(3).mean()
    df[f"{col}_roll_std_3h"] = df[col].rolling(3).std().fillna(0.0)
    df[f"{col}_roll_mean_6h"] = df[col].rolling(6).mean()

for h in HORIZONS:
    df[f"Target_t{h}"] = df["Visibility_km"].shift(-h)

# Clean all NaNs, Infs, and unused columns
df = df.replace([np.inf, -np.inf], np.nan).dropna()

target_names = [f"Target_t{h}" for h in HORIZONS]
drop_cols = ["Dew Point Temp_C"] + target_names
features = [c for c in df.columns if c not in drop_cols]

X = df[features].astype(np.float32)
y = df[target_names].astype(np.float32)

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# ---------------------------------------------------------------------------
# 3. Model Training
# ---------------------------------------------------------------------------
reg_models = {h: {} for h in HORIZONS}
clf_models = {}

reg_kwargs = dict(n_estimators=180, max_depth=5, learning_rate=0.06,
                  subsample=0.85, colsample_bytree=0.85, random_state=42)
clf_kwargs = dict(n_estimators=200, max_depth=5, learning_rate=0.07,
                  subsample=0.85, colsample_bytree=0.85, random_state=42,
                  eval_metric="mlogloss")

for h in HORIZONS:
    yh = y_train[f"Target_t{h}"]
    
    for q in QUANTILES:
        try:
            model = XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, **reg_kwargs)
            model.fit(X_train, yh)
        except Exception:
            model = XGBRegressor(objective="reg:squarederror", **reg_kwargs)
            model.fit(X_train, yh)
        reg_models[h][q] = model

    yh_cls = np.where(yh <= FOG_THRESHOLD_CRITICAL, 2,
             np.where(yh <= FOG_THRESHOLD_ADVISORY, 1, 0))
    clf = XGBClassifier(**clf_kwargs)
    weights = compute_sample_weight("balanced", yh_cls)
    clf.fit(X_train, yh_cls, sample_weight=weights)
    clf_models[h] = clf

# ---------------------------------------------------------------------------
# 4. Save Artifacts
# ---------------------------------------------------------------------------
joblib.dump({"reg": reg_models, "clf": clf_models}, "fog_models_v2.joblib")
joblib.dump({
    "features": features,
    "crit_thresh": FOG_THRESHOLD_CRITICAL,
    "adv_thresh": FOG_THRESHOLD_ADVISORY,
    "quantiles": QUANTILES,
    "horizons": HORIZONS,
    "class_labels": CLASS_LABELS,
}, "artifacts_v2.joblib")
X_test.to_csv("test_features_v2.csv")

print("Pipeline executed successfully!")

Pipeline executed successfully!
